# 1. Data exploration
What does the purchase log look like, and what does that imply for modelling?
Key questions: how sparse is it, how much do customers *re*-buy, how skewed is popularity, and how
fast does demand drift week to week (which decides how recent the history must be).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from hmrec import workspace
pd.set_option("display.width", 140); plt.rcParams["figure.dpi"] = 110

# 10% of customers, seeded. Falls back to a tiny synthetic fixture (with a warning) if the
# real data has not been downloaded yet.
W = workspace.setup(sample_frac=0.10, seed=0)
hm, split = W.hm, W.split
if W.synthetic:
    print("*** SYNTHETIC DATA - numbers below are not H&M results ***")

In [ ]:
from hmrec.data import describe
pd.Series(describe(hm)).to_frame("value")

## Purchases per week

In [ ]:
inter = hm.interactions.copy()
inter["day"] = inter["ts"] // 86400
last = inter["day"].max()
inter["weeks_ago"] = (last - inter["day"]) // 7
weekly = inter.groupby("weeks_ago").size().sort_index(ascending=False)
ax = weekly.plot(figsize=(10, 3), title="Purchases per week (right edge = test week)")
ax.set_xlabel("weeks before the end of the data"); plt.show()

## Customer activity and repurchasing

In [ ]:
per_cust = inter.groupby("user").size()
print(per_cust.describe().round(2).to_string())
pairs = inter.groupby(["user", "item"]).size()
repeat_share = (pairs > 1).sum() / len(pairs)
bought_before = inter.sort_values("ts").duplicated(["user", "item"]).mean()
print(f"share of (customer, article) pairs bought more than once: {repeat_share:.1%}")
print(f"share of ALL purchases that are a repeat of an earlier purchase: {bought_before:.1%}")
per_cust.clip(upper=per_cust.quantile(0.99)).hist(bins=50, figsize=(8, 3)); plt.title("Purchases per customer"); plt.show()

**Reading:** if a large share of purchases are repeats, a model that only recommends
*new* articles throws away its best signal - which is why the evaluator in this project does not
mask articles a customer already owns.

## Popularity is a long tail

In [ ]:
pop = inter.groupby("item").size().sort_values(ascending=False)
cum = pop.cumsum() / pop.sum()
for q in (0.01, 0.05, 0.2):
    n = int(len(hm.items) * q)
    print(f"top {q:.0%} of articles ({n}) account for {cum.iloc[min(n, len(cum)) - 1]:.1%} of purchases")
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(np.arange(len(pop)), pop.to_numpy()); ax[0].set_yscale("log"); ax[0].set_title("articles ranked by purchases (log y)")
ax[1].plot(np.arange(len(cum)) / len(hm.items), cum.to_numpy()); ax[1].set_title("cumulative share of purchases"); plt.show()

## Demand drift: how long does 'popular' stay popular?

In [ ]:
top_last_week = set(inter[inter.weeks_ago == 0].groupby("item").size().nlargest(50).index)
rows = []
for w in range(1, min(12, int(inter.weeks_ago.max())) + 1):
    top_w = set(inter[inter.weeks_ago == w].groupby("item").size().nlargest(50).index)
    rows.append((w, len(top_last_week & top_w) / 50))
pd.DataFrame(rows, columns=["weeks_before", "overlap_of_top50_with_last_week"]).set_index("weeks_before").plot(
    kind="bar", figsize=(8, 3), legend=False, title="Overlap of weekly top-50 with the final week's top-50"); plt.show()

## Who do we evaluate?

In [ ]:
print(f"validation week: {len(W.ev_val.users):,} customers with purchases and prior history")
print(f"test week:       {len(W.ev_test.users):,} customers with purchases and prior history")
print(f"split sizes (purchases): train={len(split.train):,} val={len(split.val):,} test={len(split.test):,}")
print("No future leakage:", "checked" if split.train.ts.max() < split.val_start <= split.val.ts.min() else "FAILED")